In [3]:
import numpy as np 
import pandas as pd 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [4]:
df = pd.read_csv("adani_stock_sequence.csv")
prices = df["Close"].values.astype(float)

# pd.read_csv(...) opens your CSV file and loads it as a table (df = "dataframe")
# df["Close"] grabs just the Close price column, ignoring the Date column
# .values converts that column into a plain NumPy array (just numbers, no table structure)
# .astype(float) makes sure every number is a decimal (float), not an integer

# At this point, prices is just: [2473.38, 2468.70, 2498.97, ..., 5323.02] — 1000 numbers.

In [5]:
prices_norm = prices / np.max(prices)

# np.max(prices) finds the single largest price in the whole dataset (say, ~6100)
# Dividing every price by that number squeezes all values into the range 0 to 1
# Why: neural networks train much better on small, similar-scale numbers. Feeding in raw values like 5323 makes the math unstable (huge gradients, slow/broken learning). This is the same idea as your word-script using small integers (0-11) — except prices aren't naturally small, so we shrink them ourselves.

In [6]:
WINDOW = 30

# Just a setting: "use the last 30 days of prices to predict day 31." You can change this number — it's not tied to anything else in the code.

In [7]:
X = []
Y = []
for i in range(len(prices_norm) - WINDOW):
    X.append(prices_norm[i:i + WINDOW])
    Y.append(prices_norm[i + WINDOW])

# This is the most important block — it turns one long list of prices into many small training examples.
    # len(prices_norm) - WINDOW — we stop 30 days before the end, because every window needs 30 days after it to grab the "answer" (the actual next day)
    # Each loop step:
        # prices_norm[i : i+WINDOW] — grabs 30 consecutive days → this becomes one input (X)
        # prices_norm[i+WINDOW] — grabs the very next day right after that window → this becomes the correct answer (Y) for that input

# Example with tiny numbers, WINDOW=3:

# prices_norm = [0.1, 0.2, 0.3, 0.4, 0.5]

# i=0: X = [0.1, 0.2, 0.3]  ->  Y = 0.4
# i=1: X = [0.2, 0.3, 0.4]  ->  Y = 0.5

# So from 1000 days of prices, you end up with 970 pairs of "these 30 days → this next day."

In [8]:
X = np.array(X).reshape(-1, WINDOW, 1)
Y = np.array(Y)

# np.array(X) — converts the list of lists into a proper NumPy array
# .reshape(-1, WINDOW, 1) — this is a required format for RNNs: (number_of_examples, timesteps, features_per_timestep)
    # -1 means "figure out how many examples there are automatically" (970, in this case)
    # WINDOW (30) = how many time steps (days) per example
    # 1 = how many numbers per day (just the price itself — 1 feature)
# Y stays a simple flat array of 970 target values (no reshaping needed since Dense expects a plain list of answers)

In [9]:
model = Sequential()
model.add(SimpleRNN(32, input_shape=(WINDOW, 1)))
model.add(Dense(1))

# Sequential() — start building an empty model, layers get stacked in the order you add them
# SimpleRNN(32, input_shape=(WINDOW, 1)) — the actual recurrent layer:
    # 32 = number of hidden units (the RNN's internal "memory" — bigger number = more capacity to learn patterns, but also more risk of overfitting)
    # input_shape=(WINDOW, 1) — tells the layer to expect 30 time steps, 1 value each (matches the reshape above)
    # This layer reads the 30 days one at a time, updating its internal memory as it goes, and outputs a single summary vector after seeing all 30 days
# Dense(1) — a plain output layer with 1 neuron, no activation function
    # No activation (unlike your word script's softmax) because we want a raw number (a price), not a probability distribution over categories

C:\Users\Priyanshu\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# optimizer="adam" — the algorithm that adjusts the model's internal numbers during training (standard, reliable default choice)
# loss="mse" — Mean Squared Error: measures how far off predictions are from actual prices, by squaring the difference (this is what training tries to minimize). Used for regression (predicting numbers), unlike your word script's crossentropy loss (used for classification/categories)
# metrics=["mae"] — Mean Absolute Error: just an extra number to watch during training, easier to interpret than MSE ("on average, how many rupees off are we?")

In [11]:
model.fit(X, Y, epochs=100)
print("Training Completed")

# model.fit(X, Y, epochs=100) — the actual training step: the model looks at all 970 (window → next day) examples, over and over, 100 times (epochs=100), gradually adjusting itself to get better at predicting Y from X
# print(...) — just a message once training finishes

Epoch 1/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0047 - mae: 0.0500
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.0474e-04 - mae: 0.0157
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.8666e-04 - mae: 0.0130
Epoch 4/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.7085e-04 - mae: 0.0126
Epoch 5/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.7214e-04 - mae: 0.0126
Epoch 6/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.5536e-04 - mae: 0.0122
Epoch 7/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.4236e-04 - mae: 0.0119
Epoch 8/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.5863e-04 - mae: 0.0123
Epoch 9/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.3580e-04 - mae: 0.0116
Epoch 10/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.4001e-04 - mae: 0.0118
Epoch 11/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.2394e-04 - mae: 0.0114
Epoch 12/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.3975e-04 - mae: 

In [12]:
test = prices_norm[-WINDOW:].reshape(1, WINDOW, 1)
prediction = model.predict(test)

# prices_norm[-WINDOW:] — grabs the last 30 days in your dataset (the most recent data you have)
# .reshape(1, WINDOW, 1) — same reshaping idea as before, but this time it's just 1 example (you're only predicting once), so the first number is 1 instead of -1
# model.predict(test) — asks the trained model: "given these last 30 days, what's day 31 likely to be?"

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step


In [13]:
predicted_price = prediction[0][0] * np.max(prices)
print("Predicted next value (normalized):", prediction[0][0])
print("Predicted next day price:", round(float(predicted_price), 2))

# prediction[0][0] — pulls the single predicted number out of the model's output (Keras always returns predictions wrapped in an array, so [0][0] unwraps it)
# * np.max(prices) — this undoes the normalization from earlier (remember, we divided by np.max(prices) at the start — now we multiply back by the same number to get a real rupee value instead of a 0-1 number)
# The two print lines just show you the raw (0-1) prediction and the real-world price prediction, side by side

Predicted next value (normalized): 0.89025307
Predicted next day price: 5422.1
